# sample1 - Ollama（Docker）でローカル LLM を動かす

Docker コンテナで動く **Ollama** を Python から操作します。  
WSL 環境を汚さずにローカル LLM を実行できます。

| ステップ | 内容 |
|----------|------|
| 0 | Docker コンテナの確認 |
| 1 | Ollama の基本操作 |
| 2 | Python から Ollama を呼び出す |
| 3 | ストリーミング出力 |
| 4 | チャット形式での会話 |
| 5 | システムプロンプトで役割を設定 |

## 事前準備

ターミナルで Ollama コンテナが起動しているか確認してください。

```bash
# コンテナの状態確認
docker ps | grep ollama

# 停止中の場合は起動
docker start ollama

# LLaMA 3 がなければダウンロード（約4GB）
docker exec -it ollama ollama pull llama3
```

## 0. Docker コンテナの確認

In [1]:
import subprocess
import ollama

# Docker コンテナの状態確認
result = subprocess.run(
    ['docker', 'ps', '--filter', 'name=ollama', '--format', '{{.Names}}\t{{.Status}}'],
    capture_output=True, text=True
)
print("Ollama コンテナ状態:")
print(result.stdout if result.stdout else '起動していません。docker start ollama を実行してください。')

Ollama コンテナ状態:
ollama	Up 43 minutes



## 1. Ollama の基本操作

In [3]:
# Python クライアントは localhost:11434 に自動接続（Docker のポートマッピングと一致）
models = ollama.list()
print("インストール済みモデル:")
for model in models['models']:
    print(f"  - {model['name']}")

インストール済みモデル:


KeyError: 'name'

## 2. Python から Ollama を呼び出す

In [4]:
response = ollama.chat(
    model='llama3',
    messages=[
        {'role': 'user', 'content': '日本語で自己紹介してください。3文以内で。'}
    ]
)

print("応答:")
print(response['message']['content'])
print("\n--- メタ情報 ---")
print(f"モデル      : {response['model']}")
print(f"生成トークン: {response['eval_count']}")

応答:
Here's a self-introduction in Japanese, within 3 sentences:

私たちはAIの仮想助手です。我々は、日本語を含む複数の言語を理解し、人と対話することができます。我々の目指すのは、より便利でフレンドリーなコミュニケーションを実現することです。

--- メタ情報 ---
モデル      : llama3
生成トークン: 79


## 3. ストリーミング出力

トークンを逐次出力します。長い応答でも待ち時間なく表示できます。

In [5]:
print("応答（ストリーミング）:")
for chunk in ollama.chat(
    model='llama3',
    messages=[{'role': 'user', 'content': '機械学習とディープラーニングの違いを簡単に説明してください。'}],
    stream=True
):
    print(chunk['message']['content'], end='', flush=True)
print()

応答（ストリーミング）:
A popular question! 😊

Machine learning (ML) and deep learning (DL) are often used interchangeably, but they're not exactly the same thing. Here's a brief explanation:

**Machine Learning (ML)**:

* A type of AI that allows machines to learn from data without being explicitly programmed.
* Uses algorithms to analyze and make decisions based on patterns in the data.
* Can involve various types of learning, such as:
	+ Supervised learning: learns from labeled data to predict new outcomes.
	+ Unsupervised learning: discovers hidden patterns or structures in unlabeled data.
	+ Reinforcement learning: learns through trial-and-error interactions with an environment.

Examples: image classification, natural language processing (NLP), recommendation systems.

**Deep Learning (DL)**:

* A subset of machine learning that focuses on neural networks with multiple layers.
* Inspired by the structure and function of the human brain's neural networks.
* Characterized by:
	+ Multiple hidd

## 4. チャット形式での会話

会話履歴を保持して多ターンの対話を行います。

In [6]:
messages = [
    {'role': 'system', 'content': 'あなたは親切な日本語アシスタントです。'},
    {'role': 'user',   'content': 'PyTorch と TensorFlow の違いは何ですか？'},
]

response = ollama.chat(model='llama3', messages=messages)
answer1 = response['message']['content']
print("1回目の応答:")
print(answer1)

# 会話を続ける
messages.append({'role': 'assistant', 'content': answer1})
messages.append({'role': 'user', 'content': '初心者にはどちらがおすすめですか？'})

response2 = ollama.chat(model='llama3', messages=messages)
print("\n2回目の応答:")
print(response2['message']['content'])

1回目の応答:
🤔

PyTorch and TensorFlow are two popular open-source machine learning libraries used for building and training artificial neural networks. While both libraries share some similarities, they have distinct differences in their design, architecture, and usage.

**1. Computational Graph:**
	* PyTorch builds a dynamic computational graph at runtime, which means that the graph is created only when you run your code.
	* TensorFlow, on the other hand, builds a static computational graph upfront, before running your code. This makes TensorFlow more suitable for large-scale deployments.

**2. Autograd:**
	* PyTorch uses autograd to automatically compute gradients, which simplifies the process of backpropagation and makes it easier to implement complex neural networks.
	* TensorFlow relies on manual gradient computation or uses its own auto-differentiation system (tf.GradientTape).

**3. Dynamic vs. Static Graph:**
	* PyTorch's dynamic graph allows for more flexibility in building and mo

## 5. システムプロンプトで役割を設定

In [7]:
personas = [
    ('優しい先生',   'あなたは小学生にもわかるように説明する優しい先生です。'),
    ('厳格な専門家', 'あなたは技術的に正確な情報のみを提供する厳格な専門家です。'),
]

question = 'ニューラルネットワークとは何ですか？'

for name, system_prompt in personas:
    response = ollama.chat(
        model='llama3',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': question}
        ]
    )
    print(f"=== {name} ===")
    print(response['message']['content'])
    print()

=== 優しい先生 ===
Nice question! 😊

So, you know how our brains can do lots of cool things like recognize faces, understand words, and make decisions? Like when you see a picture of your favorite animal, you can tell what it is just by looking at it?

Well, Neural Networks are kind of like special computers that try to work the same way as our brains. They're made up of many tiny "building blocks" called "neurons" (like tiny brain cells) that talk to each other and help figure out answers.

Imagine you have a bunch of boxes with lines connecting them. Each box has some information, like a picture or a word. When you put something into one box, the line connects it to another box, which then sends the information to yet another box... and so on!

Neural Networks do this too! They take in information (like a picture), process it, and send it to other "boxes" (called "layers") that try to make sense of it. As they work together, the network gets better at recognizing patterns, like shapes or 